# NeuroMind Phase 1 — Binary speech training
Train speech with Non-Dementia and Dementia labels. A held-out result is required before clinical claims. The notebook has not been rerun in this ZIP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, shutil, torch
NEUROMIND = Path('/content/drive/MyDrive/NeuroMind ')
CODE_ZIP = NEUROMIND / '12_Web_Application/NeuroMind_PHASE1_TRAINING_CODE.zip'
PROJECT_ROOT = Path('/content/neuromind_phase1')
assert NEUROMIND.is_dir(), f'NeuroMind Drive folder not found: {NEUROMIND}'
assert CODE_ZIP.is_file(), f'Training package not found: {CODE_ZIP}'
if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
shutil.unpack_archive(str(CODE_ZIP), str(PROJECT_ROOT))
%cd $PROJECT_ROOT
!pip -q install -r training/requirements-phase1.txt
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('GPU is not enabled. Select a T4 GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# The uploaded Drive folder intentionally includes a trailing space.
AUDIO_ROOT = NEUROMIND / '01_Datasets/Speech Dataset '
SEQUENCE_DIR = NEUROMIND / '03_Features/final_speech_sequences'
SPEECH_OUTPUT = NEUROMIND / '04_Models/Speech_Final'
SPEECH_LABEL_MAP = NEUROMIND / '03_Features/speech_label_map.audited.json'
SPEECH_LABEL_MAP.parent.mkdir(parents=True, exist_ok=True)
SPEECH_OUTPUT.mkdir(parents=True, exist_ok=True)
SPEECH_LABEL_MAP.write_text(json.dumps({'Non Dementia':'Non-Dementia','Dementia':'Dementia'}, indent=2))
assert (AUDIO_ROOT / 'Non Dementia').is_dir(), 'Non Dementia speech folder is missing'
assert (AUDIO_ROOT / 'Dementia').is_dir(), 'Dementia speech folder is missing'
print('Audio root:', AUDIO_ROOT)
print('Output:', SPEECH_OUTPUT)
print('Labels: Non-Dementia, Dementia (real Drive folders)')

In [ ]:
!python -m training.prepare_speech_embeddings --audio-root "$AUDIO_ROOT" --output-dir "$SEQUENCE_DIR" --label-map "$SPEECH_LABEL_MAP" --resume

In [ ]:
!python -m training.train_speech --manifest "$SEQUENCE_DIR/speech_sequence_manifest.csv" --output-dir "$SPEECH_OUTPUT" --epochs 60 --batch-size 8 --patience 10

In [ ]:
artifact = SPEECH_OUTPUT / 'neuromind_speech_bilstm_attention.pt'
manifest = Path(str(artifact) + '.manifest.json')
assert artifact.is_file() and manifest.is_file(), 'Speech artifact export failed'
metadata = json.loads(manifest.read_text())
assert metadata['class_labels'] == ['Non-Dementia', 'Dementia']
print(json.dumps({'artifact':str(artifact),'classes':metadata['class_labels'],'test_metrics':metadata['test_metrics'],'sha256':metadata['artifact_sha256']}, indent=2))

## Fusion data gate
The fusion model is binary. Train only when real speech, memory, attention and lifestyle records correspond to the same patients, with the same feature definitions as the website. Synthetic auxiliary tables are insufficient to validate screening performance.